# Flat-band spectroscopy checks (corrected)

Corrected version of *NB_O4_not_sure_runs_gemini.ipynb*.

- **Cell 0** -- CRT exact-integer reconstruction for tromino weights (unchanged; correct).
- **Cell 1** -- *Rewritten.* The original left `H_4` as a `pass` placeholder (zero matrix), so it printed "GAUSS-LAW PROTECTION CONFIRMED" for any input -- a vacuous gate. This version builds the real per-class Bloch operators `T_g(k)`, uses the true symbol `u_j = 1 - e^{i k_j}`, evaluates paper eq. (19), fails loudly on a vacuous operator, and self-tests that it discriminates flat from non-flat.
- **Cell 2** -- O(y^2) C-even sanity checks (unchanged; constants match the paper exactly).

Requires `numpy` and `sympy`.


In [1]:
import json
import fractions

def crt_reconstruct(mod_evals):
    """
    Reconstructs the exact integer from its modular evaluations using the
    Chinese Remainder Theorem.
    Includes a half-modulus shift to correctly reconstruct negative integers.
    """
    primes = [int(p) for p in mod_evals.keys()]
    rems = [int(r) for r in mod_evals.values()]

    # Calculate global modulus M
    M = 1
    for p in primes:
        M *= p

    total = 0
    for p, r in zip(primes, rems):
        M_i = M // p
        # Compute modular inverse using Python 3.8+ built-in
        y_i = pow(M_i, -1, p)
        total = (total + r * M_i * y_i) % M

    # Correct for negative integers (values strictly in the upper half of the finite field)
    if total > M // 2:
        total -= M

    return total

def assemble_final_weights(gpu_results):
    """
    Combines the CRT-reconstructed tensor integer with the exact rational
    scalar from Phase 1 to produce the final tromino weight w_g.
    """
    final_weights = {}

    for class_id, data in gpu_results.items():
        # Step 1: Reconstruct the exact tensor integer via CRT
        exact_tensor_int = crt_reconstruct(data['mod_evals'])

        # Step 2: Multiply by the exact scalar from Phase 1
        scalar = fractions.Fraction(data['numerator'], data['denominator'])
        final_weight = exact_tensor_int * scalar

        final_weights[class_id] = {
            'exact_tensor_int': exact_tensor_int,
            'final_weight_fraction': str(final_weight),
            'final_weight_float': float(final_weight)
        }

    return final_weights

if __name__ == "__main__":
    # Ingest Phase 2 output
    gpu_output = {
      "word_001": {
        "numerator": 3,
        "denominator": 64,
        "mod_evals": {
          "2147483647": 1,
          "2147483629": 1,
          "2147483587": 1
        }
      }
    }

    # Execute Phase 3 Reconstruction
    final_results = assemble_final_weights(gpu_output)
    print(json.dumps(final_results, indent=2))

{
  "word_001": {
    "exact_tensor_int": 1,
    "final_weight_fraction": "3/64",
    "final_weight_float": 0.046875
  }
}


In [2]:
# Cell 1 (REWRITTEN) -- Paper eq. (19) fourth-order flatness criterion, evaluated for REAL.
#
# WHY THE ORIGINAL CELL WAS WRONG:
#   It built H_4 with a `pass` placeholder, so H_4 stayed the ZERO matrix and
#   u^dag H_4 P_perp = 0 trivially. It therefore printed "GAUSS-LAW PROTECTION
#   CONFIRMED" for ANY input (even empty) -- a vacuous gate that tested nothing.
#
# WHAT THIS DOES INSTEAD:
#   Assembles the REAL per-geometry-class Bloch operators T_g(k) from the certified
#   tromino suite, uses the true cube-boundary flat state u_j = 1 - e^{i k_j}, and
#   evaluates the paper's criterion
#       flat <=> u(k)^dag H_4(k) P_perp(k) == 0  (eq. 19),  H_4 = sum_g w_g T_g(k)
#   equivalently "H_4(k) maps the flat state to a multiple of itself". It HARD-FAILS
#   on a vacuous H_4, and a self-test proves the criterion discriminates flat/non-flat.
#
# HONESTY (verified this session): no physical SU(3) weights are stored anywhere in
#   THEORY / E:YANG (searched). Only candidate weights (W_path+/- = 2/9, W_corner =
#   2/27, from Untitled221) exist; they are an unnormalized hypothesis, not physical.
#   This cell defaults to them and labels the verdict conditional; it does NOT settle
#   Conjecture 7.4.

import itertools
from collections import Counter, defaultdict
import numpy as np

ORIENT = [(0, 1), (0, 2), (1, 2)]
LIFTERS = ["path_bent_or_straight_prod+1", "path_bent_or_straight_prod-1", "triangle_corner_cyc-1"]

CANDIDATE_WEIGHTS = {
    "path_bent_or_straight_prod+1": 2 / 9,
    "path_bent_or_straight_prod-1": 2 / 9,
    "triangle_corner_cyc-1":        2 / 27,
}


def _shift(x, d, L, n=1):
    z = list(x); z[d] = (z[d] + n) % L; return tuple(z)

def _cdisp(x, y, L):
    return tuple(((y[i] - x[i] + L // 2) % L) - L // 2 for i in range(3))

def _boundary(x, o, L):
    mu, nu = ORIENT[o]; x = tuple(x)
    return [((x, mu), +1), ((_shift(x, mu, L), nu), +1), ((_shift(x, nu, L), mu), -1), ((x, nu), -1)]

def _build_complex(L):
    plaqs = [(x, o) for x in itertools.product(range(L), repeat=3) for o in range(3)]
    inc = defaultdict(list)
    for i, p in enumerate(plaqs):
        for ln, sg in _boundary(p[0], p[1], L):
            inc[ln].append((i, sg))
    nbrs = {i: {} for i in range(len(plaqs))}
    for ln, lst in inc.items():
        for (i, si), (j, sj) in itertools.combinations(lst, 2):
            nbrs[i][j] = (si * sj, ln); nbrs[j][i] = (si * sj, ln)
    return plaqs, nbrs

def _classify(i, q, r, nbrs, bsets):
    if r == i:
        return "backtrack_2plaquette"
    s1 = nbrs[i][q][0]; s2 = nbrs[q][r][0]; prod = s1 * s2
    com3 = bsets[i] & bsets[q] & bsets[r]
    if r in nbrs[i]:
        cyc = s1 * s2 * nbrs[r][i][0]
        return "triangle_same_link_cyc%+d" % cyc if len(com3) == 1 else "triangle_corner_cyc%+d" % cyc
    return "path_common_link_prod%+d" % prod if len(com3) == 1 else "path_bent_or_straight_prod%+d" % prod

def build_class_terms(L=6):
    plaqs, nbrs = _build_complex(L)
    bsets = [set(ln for ln, _ in _boundary(x, o, L)) for x, o in plaqs]
    terms = defaultdict(Counter)
    for i in range(len(plaqs)):
        xi, oi = plaqs[i]
        for q, (s1, _) in nbrs[i].items():
            for r, (s2, _) in nbrs[q].items():
                cls = _classify(i, q, r, nbrs, bsets)
                xr, orr = plaqs[r]; d = _cdisp(xi, xr, L)
                terms[cls][(oi, orr, d, s1 * s2)] += 1
    return terms

def T_g(counter, k, L):
    """Per-geometry-class Bloch matrix T_g(k) (the real 'T_g' the paper's H_4 sums)."""
    M = np.zeros((3, 3), dtype=complex)
    for (o, oj, d, s), m in counter.items():
        M[o, oj] += (m / L ** 3) * s * np.exp(1j * np.dot(k, d))
    return M

def flat_state(k):
    """Cube-boundary null vector built from the true symbol u_j = 1 - e^{i k_j}."""
    u = [1 - np.exp(1j * q) for q in k]
    return np.array([np.conjugate(u[2]), -np.conjugate(u[1]), np.conjugate(u[0])], dtype=complex)

def H4(terms, weights, k, L):
    M = np.zeros((3, 3), dtype=complex)
    for cls, c in terms.items():
        M += weights.get(cls, 0.0) * T_g(c, k, L)
    return M

def eq19_residual(terms, weights, ks, L):
    """max ||P_perp H_4 w|| / ||H_4 w|| over sampled k; == 0 iff eq.(19) holds."""
    worst = 0.0
    for k in ks:
        w = flat_state(k); nrm = np.vdot(w, w).real
        if nrm < 1e-12:
            continue
        Mw = H4(terms, weights, k, L) @ w
        alpha = np.vdot(w, Mw) / nrm
        worst = max(worst, np.linalg.norm(Mw - alpha * w) / (np.linalg.norm(Mw) + 1e-12))
    return worst

def evaluate_fourth_order_criterion(weights, L=6, n_k=96, tol=1e-9):
    terms = build_class_terms(L)
    rng = np.random.default_rng(20260611)
    ks = rng.uniform(0.19, 2 * np.pi - 0.19, size=(n_k, 3))
    Hnorm = max(np.linalg.norm(H4(terms, weights, k, L)) for k in ks)
    assert Hnorm > 1e-12, "VACUOUS H_4 (zero operator): weights/geometry not wired in -- refusing to certify."
    resid = eq19_residual(terms, weights, ks, L)
    return (resid < tol), resid

def _self_test():
    """Proves the criterion has teeth: equal-lifter weights are flat; candidate weights are not."""
    f_flat, r_flat = evaluate_fourth_order_criterion({c: 1.0 for c in LIFTERS})
    f_cand, r_cand = evaluate_fourth_order_criterion(dict(CANDIDATE_WEIGHTS))
    assert f_flat and not f_cand, (
        "criterion not discriminating: equal-lifter flat=%s(r=%.1e), candidate flat=%s(r=%.1e)"
        % (f_flat, r_flat, f_cand, r_cand))
    return r_flat, r_cand


if __name__ == "__main__":
    r_flat, r_cand = _self_test()
    print("Self-test (proves the eq.(19) test discriminates, unlike the old placeholder):")
    print("  equal-lifter weights -> residual %.2e  => FLAT (protected)   [expected]" % r_flat)
    print("  candidate weights    -> residual %.2e  => LIFTS             [expected]" % r_cand)
    print()
    print("Evaluating eq.(19)  u(k)^dag H_4(k) P_perp(k) == 0  over the Brillouin zone,")
    print("H_4 = sum_g w_g T_g(k), with the CANDIDATE weights (W_path+/- = 2/9, W_corner = 2/27):")
    is_flat, resid = evaluate_fourth_order_criterion(CANDIDATE_WEIGHTS)
    print("  max flat-line residual = %.3e" % resid)
    if is_flat:
        print("  RESULT (conditional on candidate weights): FLAT -- band stays exactly flat.")
    else:
        print("  RESULT (conditional on candidate weights): BANDWIDTH ACQUIRED -- band is lifted.")
    print()
    print("STATUS (verified this session, not parroting the suite's note):")
    print("  * Candidate weights are an UNNORMALIZED primitive hypothesis (Untitled221), not physical;")
    print("    no physical weights are stored anywhere in THEORY / E:YANG (searched) -> a compute task.")
    print("  * The whole question reduces to ONE equality: is Wp == Wc ?  (flat iff equal; closed form")
    print("    alpha(k) = 12*Wp - 4*(Wp-Wc)*e2/e1).  By C-symmetry W_path+ = W_path- = Wp.")
    print("  * The DIRECT O(y^3) part is exactly zero (all triple-trace Haar moments vanish), so Wp-Wc")
    print("    is a pure reduced-resolvent effect over channels {1,8,3bar,6} at energies {4,11/2,14/3,17/3};")
    print("    computing it (calibrated to t_+ = -11/306) is the remaining step.")


Self-test (proves the eq.(19) test discriminates, unlike the old placeholder):
  equal-lifter weights -> residual 4.93e-16  => FLAT (protected)   [expected]
  candidate weights    -> residual 2.71e-01  => LIFTS             [expected]

Evaluating eq.(19)  u(k)^dag H_4(k) P_perp(k) == 0  over the Brillouin zone,
H_4 = sum_g w_g T_g(k), with the CANDIDATE weights (W_path+/- = 2/9, W_corner = 2/27):
  max flat-line residual = 2.712e-01
  RESULT (conditional on candidate weights): BANDWIDTH ACQUIRED -- band is lifted.

STATUS (verified this session, not parroting the suite's note):
  * Candidate weights are an UNNORMALIZED primitive hypothesis (Untitled221), not physical;
    no physical weights are stored anywhere in THEORY / E:YANG (searched) -> a compute task.
  * The whole question reduces to ONE equality: is Wp == Wc ?  (flat iff equal; closed form
    alpha(k) = 12*Wp - 4*(Wp-Wc)*e2/e1).  By C-symmetry W_path+ = W_path- = Wp.
  * The DIRECT O(y^3) part is exactly zero (all triple-trac

In [3]:
import sympy as sp
import fractions

def execute_ceven_sanity_checks():
    """
    Evaluates the C-even band structure at O(y^2) using the corrected hop t_+ = -11/306.
    Asserts exact values for band bottom, top, bandwidth, curvature, and the E^{++} state.
    """
    y, k_mag = sp.symbols('y k_mag', real=True)

    # Exact constants from the O(y^2) assembly
    tower_y2 = sp.Rational(13, 20)
    vacuum_sub = sp.Rational(3, 4)
    self_energy = sp.Rational(-481, 612)

    # Corrected hop: t_+ = -481/612 + 3/4 = -11/306
    t_plus = self_energy + vacuum_sub

    # Diagonal coefficient: 12 neighbors * (-481/612 + 3/4)
    diag_leakage = 12 * t_plus

    # Base constant for y^2 term: tower + diag_leakage
    base_y2 = tower_y2 + diag_leakage

    # The C-even band polynomial at O(y^2) (ignoring 8/3 - y static/tower terms for width)
    # E_+(k, y) = y^2 * [ 223/1020 - 11/306 * \lambda(k) ]
    def E_plus_y2(lam):
        return base_y2 + t_plus * lam

    print("--- C-EVEN O(y^2) SANITY CHECKS ---")

    # 1. Band Bottom (A_1^{++} state) at k = 0 -> \lambda = 12
    bottom = E_plus_y2(12)
    expected_bottom = sp.Rational(-217, 1020)
    assert bottom == expected_bottom, f"Band bottom failed: {bottom} != {expected_bottom}"
    print(f"[PASS] Band Bottom (A_1^{{++}}): {bottom}")

    # 2. Band Top at zone faces -> \lambda = -4
    top = E_plus_y2(-4)
    expected_top = sp.Rational(1109, 3060)
    assert top == expected_top, f"Band top failed: {top} != {expected_top}"
    print(f"[PASS] Band Top: {top}")

    # 3. Bandwidth
    bandwidth = top - bottom
    expected_width = sp.Rational(88, 153) # 16 * |t_+|
    assert bandwidth == expected_width, f"Bandwidth failed: {bandwidth} != {expected_width}"
    print(f"[PASS] Exact Bandwidth: {bandwidth} (16 * |t_+|)")

    # 4. Hop-independent E^{++} doublet -> \lambda = 0
    e_plus_plus = E_plus_y2(0)
    expected_e = sp.Rational(223, 1020)
    assert e_plus_plus == expected_e, f"E^{{++}} level failed: {e_plus_plus} != {expected_e}"
    print(f"[PASS] E^{{++}} Level: {e_plus_plus}")

    # 5. Effective Mass and Curvature
    # \lambda(k) expands isotropically as 12 - (4/3)|k|^2
    lam_expansion = 12 - sp.Rational(4, 3) * k_mag**2
    energy_dispersion = E_plus_y2(lam_expansion)

    # Curvature is the coefficient of k_mag^2
    curvature = energy_dispersion.coeff(k_mag, 2)
    expected_curvature = sp.Rational(22, 459)
    assert curvature == expected_curvature, f"Curvature failed: {curvature} != {expected_curvature}"

    # Effective mass m* = (2 * curvature * y^2)^{-1}
    # In units where we isolate the numeric fraction: m* = 1 / (2 * curvature)
    eff_mass_coeff = 1 / (2 * curvature)
    expected_mass_coeff = sp.Rational(459, 44)
    assert eff_mass_coeff == expected_mass_coeff, f"Mass failed: {eff_mass_coeff} != {expected_mass_coeff}"
    print(f"[PASS] Curvature: +{curvature} y^2")
    print(f"[PASS] Effective Mass: {eff_mass_coeff} / y^2")

if __name__ == "__main__":
    execute_ceven_sanity_checks()

--- C-EVEN O(y^2) SANITY CHECKS ---
[PASS] Band Bottom (A_1^{++}): -217/1020
[PASS] Band Top: 1109/3060
[PASS] Exact Bandwidth: 88/153 (16 * |t_+|)
[PASS] E^{++} Level: 223/1020
[PASS] Curvature: +22/459 y^2
[PASS] Effective Mass: 459/44 / y^2
